In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import torch
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler, DataCollatorWithPadding, TrainingArguments, Trainer
from torch.optim import AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import evaluate

## Load Dataset

In [ ]:
from datasets import load_dataset

red_pajama = load_dataset("togethercomputer/RedPajama-Data-V2", 'sample', split="train")

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

In [4]:
red_pajama

DatasetDict({
    train: Dataset({
        features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
        num_rows: 1050391
    })
})

In [5]:
red_pajama = red_pajama.train_test_split(test_size=0.3)
red_pajama

AttributeError: 'DatasetDict' object has no attribute 'train_test_split'

In [6]:
training_samples = red_pajama['train']
validation_samples = red_pajama['test']
training_samples = training_samples.rename_column('raw_content', 'text')
validation_samples = validation_samples.rename_column('raw_content', 'text')
print(training_samples)
print(validation_samples)

Dataset({
    features: ['text', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 735273
})
Dataset({
    features: ['text', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 315118
})


In [7]:
training_samples[0]

{'text': 'Photo: AFP via Getty Images\nSan Francisco In-N-Out Temporarily Closed for Not Requiring Proof of Vaccination\n~ 3 minutes read\nLast week, the San Francisco Department of Public Health temporarily shut down the city’s In-n-Out location after employees failed to ask customers for proof of vaccination — a requirement for indoor dining in San Francisco.\nThe restaurant has since re-opened without the option of indoor dining.\nIn-n-Out chief legal and business officer Arnie Wensinger made the following statement: “We refuse to become the vaccination police for any government. It is unreasonable, invasive, and unsafe to force our restaurant Associates to segregate Customers into those who may be served and those who may not.”\nWhat else is each side focusing on?\nMandating vaccinations to participate in unnecessary indoor activities saves lives.\nThe Delta variant can make even vaccinated people sick, especially if their immune systems are weak.\nMandating vaccinations for everyd

In [8]:
validation_samples[0]

{'text': 'Wählen Sie Anschluss\nSchwulen Treffen\nMollige Singles\nCraigslist Kontaktanzeigen\nReife Frauen treffen\nSinglebörse MILF\nFrauen Suchen Sex\nSinglebörse Lesben\nAnschluss-Apps\nAsian dating app\nApps für Studenten\niOS-Anschluss-Apps\nLesbische App\nTeenager Apps\nSchwarze Apps\nApps für Paare\nSuche Affäre\nhttps://besthookupwebsites.org/de/flirt-review/Flirt\nFlirt im Test 2023\nÜber Site\nAktives Publikum 68%\nQualitätsübereinstimmungen 92%\nBeliebtes Alter 23-24\nAntwortquote 90%\nBenutzerfreundlichkeit 6\nPopularität 9.2\nBetrug Sehr selten\nAnmeldung Kostenlos\nFlirt Benutzer melden sich auch hier an:\n18 Nov 2020Aktualisiert:18 Jan 2023\n2862 Ansichten\nBeste Adult Dating Sites\nDating-Sites für Erwachsene Erotische Websites\nBeste Dating-Sites nach Beziehungstyp\nHookup Gelegenheitssex\nVorteile unt Nachteile\nFlirt bietet eine einfache und schnelle Registrierung in wenigen Schritten kostenlos. Ihr Ausweis oder andere Dokumente sind nicht erforderlich.\nLike Galler

In [9]:
training_samples.column_names

['text', 'doc_id', 'meta', 'quality_signals']

In [10]:
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# to help save on gpu space and run this a bit faster we'll load the model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

In [13]:
from peft import LoraConfig
# rank defines the rank of the adapter matrix,
# the higher the rank, the more complex the task it's trying to learn
rank = 128

# the alpha is a scaling factor hyper parameter, basically controls how much our
# adapter will influence the models output, the higher this value
# the more our adapter will overpower the original model weights.
# there is a lot of advice out there for what the alpha value should be
# keeping the alpha at around 2x of what the rank is works for this notebook
alpha = rank*2
peft_config = LoraConfig(
    r=rank,
    lora_alpha=alpha,
    lora_dropout=0.05, # dropout for the lora layers while training, to avoid overfitting
    bias="none",
    task_type="CAUSAL_LM",
    # the target modules defines what types of layers to add lora adapters too, so in the network
    # any model that have a name in this list will have a lora adapter added to it,
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj', 'gate_proj', 'down_proj', 'up_proj']
)

In [17]:
from transformers import TrainingArguments
from trl import SFTTrainer

model_checkpoint_path = "./results/llama-7b"

# an important note is that the loss function isn't defined here,
# it's instead stored as a model parameter for models in hf,
# in the case of llama it is cross entropy loss

# first define some training arguments
training_arguments = TrainingArguments(
    output_dir=model_checkpoint_path,
    optim='adamw_torch', #specify what optimizer we wwant to use, in this case a 8bit version of adamw with pagination.
    per_device_train_batch_size=16, # define the number of samples per training batch
    gradient_accumulation_steps=8, # define how many steps to accumulate gradients,
    log_level='debug',
    eval_strategy = "steps",
    save_strategy='steps', # we'll save a checkpoint every epoch
    logging_steps=20,
    eval_steps=40,
    save_steps=40,
    learning_rate=1e-5, # for llm training we want a fairly high learning rate, 1e-4 is a good starting point but it's worth it to play around with this value
    fp16=True,
    num_train_epochs=4,
    max_steps=120,
    save_total_limit=2,
    warmup_ratio=0.1,
    load_best_model_at_end = True,
    overwrite_output_dir = True,
    lr_scheduler_type='linear',# and set our learning rate decay
)

# now that we have our arguments, we'll use that to create our trainer,
# passing in the model, dataset, peft config, tokenizer, ect
trainer = SFTTrainer(
    model=model,
    train_dataset=training_samples,
    eval_dataset=validation_samples,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
PyTorch: setting up devices
PyTorch: setting up devices
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [18]:
trainer.model.print_trainable_parameters()

trainable params: 319,815,680 || all params: 7,058,231,296 || trainable%: 4.5311


In [19]:
trainer.train()

Currently training with a batch size of: 8
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: quality_signals, text, meta, doc_id. If quality_signals, text, meta, doc_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 735,273
  Num Epochs = 1
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 4
  Total optimization steps = 10
  Number of trainable parameters = 319,815,680
/home/hans/miniconda3/envs/llm/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you 

Step,Training Loss,Validation Loss
8,1.825400,nan


The following columns in the evaluation set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: quality_signals, text, meta, doc_id. If quality_signals, text, meta, doc_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 315118
  Batch size = 8
Saving model checkpoint to ./results/llama-7b/checkpoint-8
loading configuration file config.json from cache at /home/hans/.cache/huggingface/hub/models--meta-llama--Llama-2-7b-hf/snapshots/01c7f73d771dfac7d292323805ebc428287df4f9/config.json
Model config LlamaConfig {
  "_name_or_path": "meta-llama/Llama-2-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 4096,
  

TrainOutput(global_step=10, training_loss=1.9972634315490723, metrics={'train_runtime': 68491.8979, 'train_samples_per_second': 0.005, 'train_steps_per_second': 0.0, 'total_flos': 1.3593414064275456e+16, 'train_loss': 1.9972634315490723})

In [ ]:
trainer.save_model("./models/llama2-7b")

In [ ]:
training_loss_history = []
eval_loss_history = [initial_eval_loss]
for step in trainer.state.log_history:
  if 'loss' in step:
    training_loss_history.append(step['loss'])
  elif "eval_loss" in step:
    eval_loss_history.append(step['eval_loss'])

print(training_loss_history)
print(eval_loss_history)

import matplotlib.pyplot as plt
time_steps = [i*16 for i in range(1, len(training_loss_history)+1)]
plt.plot(time_steps, training_loss_history, label="train loss")
plt.plot([0]+time_steps, eval_loss_history, label="eval loss")
plt.title("Train and Eval Loss During Training")
plt.xlabel("Training Step")
plt.ylabel("Cross Entropy Loss")
plt.legend(loc="upper right")
plt.show()

In [ ]:
modelpath = "./models/llama2-7b"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    modelpath,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(modelpath)

In [ ]:
from peft import PeftModel, PeftConfig

base_model_name = "meta-llama/Llama-2-7b-hf"
trained_adapter_dir = modelpath  # your checkpoint folder clearly stated here

# Load base tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Clearly load base model explicitly
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Explicitly load your trained LoRA adapters clearly
model = PeftModel.from_pretrained(base_model, trained_adapter_dir)

# Set tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model.to("cuda")
model.eval()

# Use model explicitly for inference clearly
prompt = "What is the highest mountain in the world"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_length=256)

response = tokenizer.decode(output.squeeze(), skip_special_tokens=True)
print(response)